In [1]:
import pandas as pd
import itertools
from collections import Counter

df = pd.read_csv('../mimic-iv_data/hosp/diagnoses_icd.csv')

In [2]:
df

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9
...,...,...,...,...,...
5006879,19999987,23865745,7,41401,9
5006880,19999987,23865745,8,78039,9
5006881,19999987,23865745,9,0413,9
5006882,19999987,23865745,10,36846,9


In [3]:
unique_codes = sorted(set(df['icd_code']))
len(unique_codes)

26143

In [6]:
vocab = {str(code): i for i, code in enumerate(unique_codes, start=4)}
vocab['[UNK]'] = 0
vocab['[CLS]'] = 1
vocab['[SEP]'] = 2
vocab['[PAD]'] = 3

import json
with open('./data/vocab.json', 'w') as f:
    json.dump(vocab, f)

In [7]:
with open('./data/icd_9_10_vocab.txt', 'w') as f:
    f.write('\n'.join(unique_codes))

In [8]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

from tokenizers.trainers import WordLevelTrainer
trainer = WordLevelTrainer(special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

In [10]:
tokenizer.train(['./data/icd_9_10_vocab.txt'], trainer)

In [11]:
tokenizer.token_to_id('[SEP]'), tokenizer.token_to_id('[CLS]')

(2, 1)

In [12]:
from tokenizers.processors import BertProcessing
tokenizer.post_processor = BertProcessing(
    sep=('[SEP]', tokenizer.token_to_id('[SEP]')), 
    cls=('[CLS]', tokenizer.token_to_id('[CLS]')))

In [14]:
tokenizer.save('./data/tokenizer-icd.json')

In [15]:
text = ' '.join(unique_codes[:200])
output = tokenizer.encode(text,)

In [18]:
from transformers import PreTrainedTokenizerFast

t = PreTrainedTokenizerFast(tokenizer_file='./data/tokenizer-icd.json')
t.add_special_tokens({'pad_token': '[PAD]', 'mask_token': '[MASK]'})

0

In [29]:
t.save_pretrained('./data/tokenizer-icd-final')

('./data/tokenizer-icd-final/tokenizer_config.json',
 './data/tokenizer-icd-final/special_tokens_map.json',
 './data/tokenizer-icd-final/tokenizer.json')

In [23]:
encoding = t(
            text,
            return_offsets_mapping=False,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_overflowing_tokens=False)

In [28]:
t(unique_codes[:12], is_split_into_words=True, padding='max_length', max_length=128)

{'input_ids': [1, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,